# 01: Data dictionary and relationship map

**Owner:** John (Data Engineering & Acquisition Analyst) · **Ticket:** #3

Profiles the three raw MyAnimeList files so the team knows what every field holds, how the files join, and which data-quality problems the cleaning step (#4) has to handle.

This notebook only *measures*. The column meanings and the "Dataset Description" report section are written by hand in `docs/data-dictionary.md`.

Run `uv run download-data` first if `data/raw/` is empty.

In [1]:
import ast
from pathlib import Path

import pandas as pd

from capstone.data import RAW_DIR

pd.set_option("display.max_colwidth", 60)

animes = pd.read_csv(RAW_DIR / "animes.csv")
profiles = pd.read_csv(RAW_DIR / "profiles.csv")
reviews = pd.read_csv(RAW_DIR / "reviews.csv")
raw = {"animes": animes, "profiles": profiles, "reviews": reviews}

pd.DataFrame({name: df.shape for name, df in raw.items()}, index=["rows", "columns"]).T

,rows,columns
animes,19311,12
profiles,81727,5
reviews,192112,7


## 1. Column profiles

For each column: type, % missing, number of distinct values, and one example. `text` example values are whitespace-collapsed for display only.

In [2]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    def example(s: pd.Series) -> str:
        v = s.dropna()
        return " ".join(str(v.iloc[0]).split())[:60] if len(v) else ""

    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_%": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
        "example": {c: example(df[c]) for c in df.columns},
    })

profiles_by_file = {name: profile(df) for name, df in raw.items()}
profiles_by_file["animes"]

,dtype,missing_%,n_unique,example
uid,int64,0.00,16216,28891
title,str,0.00,16214,Haikyuu!! Second Season
synopsis,str,5.05,15194,"Following their participation at the Inter-High, the Kar..."
genre,str,0.00,4851,"['Comedy', 'Sports', 'Drama', 'School', 'Shounen']"
aired,str,0.00,11127,"Oct 4, 2015 to Mar 27, 2016"
episodes,float64,3.66,199,25.0
members,int64,0.00,8803,489888
popularity,int64,0.00,14382,141
ranked,float64,16.63,14422,25.0
score,float64,3.00,580,8.82


In [3]:
profiles_by_file["profiles"]

,dtype,missing_%,n_unique,example
profile,str,0.00,47885,DesolatePsyche
gender,str,34.10,3,Male
birthday,str,42.73,7708,"Oct 2, 1994"
favorites_anime,str,0.00,35395,"['33352', '25013', '5530', '33674', '1482', '269', '1824..."
link,str,0.00,47885,https://myanimelist.net/profile/DesolatePsyche


In [4]:
profiles_by_file["reviews"]

,dtype,missing_%,n_unique,example
uid,int64,0.0,130519,255938
profile,str,0.0,47885,DesolatePsyche
anime_uid,int64,0.0,8113,34096
text,str,0.0,130440,more pics Overall 8 Story 8 Animation 8 Sound 10 Charact...
score,int64,0.0,12,8
scores,str,0.0,29837,"{'Overall': '8', 'Story': '8', 'Animation': '8', 'Sound'..."
link,str,0.0,130519,https://myanimelist.net/reviews.php?id=255938


## 2. Duplicates

Exact duplicate rows, and duplicates on each file's natural key (`animes.uid`, `profiles.profile`, `reviews.uid`). A key duplicate that isn't an exact duplicate means the same record was scraped twice with different values.

In [5]:
keys = {"animes": "uid", "profiles": "profile", "reviews": "uid"}

pd.DataFrame({
    name: {
        "rows": len(df),
        "exact_duplicate_rows": df.duplicated().sum(),
        f"duplicate_keys": df.duplicated(subset=keys[name]).sum(),
        "unique_keys": df[keys[name]].nunique(),
    }
    for name, df in raw.items()
}).T

,rows,exact_duplicate_rows,duplicate_keys,unique_keys
animes,19311,2943,3095,16216
profiles,81727,33825,33842,47885
reviews,192112,61593,61593,130519


In [6]:
# Key duplicates that are NOT exact duplicates: rows that disagree with each other
for name, df in raw.items():
    dedup = df.drop_duplicates()
    conflicting = dedup[dedup.duplicated(subset=keys[name], keep=False)]
    print(f"{name}: {conflicting[keys[name]].nunique()} keys with conflicting rows")

animes: 152 keys with conflicting rows
profiles: 17 keys with conflicting rows


reviews: 0 keys with conflicting rows


In [7]:
# For conflicting keys, which columns actually differ between the copies?
for name, df in raw.items():
    dedup = df.drop_duplicates()
    conflicting = dedup[dedup.duplicated(subset=keys[name], keep=False)]
    if conflicting.empty:
        continue
    differs = conflicting.groupby(keys[name]).nunique(dropna=False).gt(1).sum()
    print(f"{name}: columns that differ, by number of conflicting keys")
    print(differs[differs > 0].to_string(), "\n")

animes: columns that differ, by number of conflicting keys
members       138
popularity     53
ranked         10 

profiles: columns that differ, by number of conflicting keys
birthday            1
favorites_anime    17 



In [8]:
# % missing once each file is reduced to one row per key (the figures used in docs/data-dictionary.md)
deduped = {name: df.drop_duplicates(subset=keys[name]) for name, df in raw.items()}
missing = pd.concat({name: (df.isna().mean() * 100).round(2) for name, df in deduped.items()})
missing[missing > 0].rename("missing_%").to_frame()  # every other column is 0% missing

missing_%
animes   synopsis       4.71
         episodes       3.03
         ranked        10.26
         score          2.10
         img_url        1.02
profiles gender        35.52
         birthday      43.93

## 3. Relationships between files

```
profiles.profile  ──<  reviews.profile
animes.uid        ──<  reviews.anime_uid
animes.uid        ──<  profiles.favorites_anime  (list of anime IDs stored as text)
```

Checks run on de-duplicated data (one row per key): how many references actually resolve to a record in the other file.

In [9]:
anime_ids = set(deduped["animes"]["uid"])
profile_names = set(deduped["profiles"]["profile"])
fav_ids = deduped["profiles"]["favorites_anime"].dropna().map(ast.literal_eval).explode().dropna().astype(int)

def match(label: str, refs: pd.Series, targets: set) -> dict:
    refs = refs.dropna()
    hit = refs.isin(targets)
    return {
        "relationship": label,
        "references": len(refs),
        "resolve_%": round(hit.mean() * 100, 2),
        "distinct_unresolved": refs[~hit].nunique(),
    }

pd.DataFrame([
    match("reviews.anime_uid → animes.uid", deduped["reviews"]["anime_uid"], anime_ids),
    match("reviews.profile → profiles.profile", deduped["reviews"]["profile"], profile_names),
    match("profiles.favorites_anime → animes.uid", fav_ids, anime_ids),
]).set_index("relationship")

,references,resolve_%,distinct_unresolved
relationship,,,
reviews.anime_uid → animes.uid,130519,100.00,0
reviews.profile → profiles.profile,130519,100.00,0
profiles.favorites_anime → animes.uid,216695,99.82,34


In [10]:
# Review coverage: how many anime have reviews at all, and how many per anime
reviews_per_anime = reviews.drop_duplicates(subset="uid").groupby("anime_uid").size()
coverage = animes.drop_duplicates(subset="uid")["uid"].isin(reviews_per_anime.index)

print(f"anime with ≥1 review: {coverage.sum()} of {len(coverage)} ({coverage.mean():.1%})")
reviews_per_anime.describe().round(1)

anime with ≥1 review: 8113 of 16216 (50.0%)


count    8113.0
mean       16.1
std        46.3
min         1.0
25%         1.0
50%         3.0
75%        11.0
max      1292.0
dtype: float64

In [11]:
# Is review coverage concentrated on popular titles?
anime_u = deduped["animes"].set_index("uid")
n_reviews = reviews_per_anime.reindex(anime_u.index, fill_value=0)
top500 = n_reviews[anime_u["popularity"].nsmallest(500).index]

print(f"Spearman(members, review count): {anime_u['members'].corr(n_reviews, method='spearman'):.3f}")
print(f"500 most popular anime: {(top500 >= 10).mean():.1%} have ≥10 reviews; median {top500.median():.1f}")

Spearman(members, review count): 0.797
500 most popular anime: 98.8% have ≥10 reviews; median 91.5


## 4. Fields that need parsing

Several columns hold structured values stored as text. Cleaning (#4) will need to parse them.

In [12]:
genres = animes["genre"].dropna().map(ast.literal_eval)
print(f"genre: stored as text lists; {genres.map(len).describe()[['min', 'mean', 'max']].round(1).to_dict()} genres per anime")
print(f"       {genres.explode().nunique()} distinct genres; empty lists: {(genres.map(len) == 0).sum()}")

score_keys = reviews["scores"].map(lambda s: tuple(ast.literal_eval(s).keys())).value_counts()
print(f"\nscores: stored as text dicts; key sets seen:")
print(score_keys.to_string())

genre: stored as text lists; {'min': 0.0, 'mean': 2.9, 'max': 13.0} genres per anime
       43 distinct genres; empty lists: 82



scores: stored as text dicts; key sets seen:
scores
(Overall, Story, Animation, Sound, Character, Enjoyment)    192112


In [13]:
# Does reviews.score equal the 'Overall' entry in reviews.scores?
overall = reviews["scores"].map(lambda s: ast.literal_eval(s).get("Overall")).astype(float)
print(f"reviews.score == scores['Overall']: {(overall == reviews['score']).mean():.2%} of rows")

# Review text: leading/trailing whitespace and length
text = deduped["reviews"]["text"].fillna("")
print(f"review text with leading whitespace: {text.str.match(r'^\s').mean():.1%}")
print(f"review length (words) after collapsing whitespace:")
text.str.split().map(len).describe().round(0)

reviews.score == scores['Overall']: 100.00% of rows


review text with leading whitespace: 100.0%
review length (words) after collapsing whitespace:


count    130519.0
mean        502.0
std         478.0
min          27.0
25%         200.0
50%         360.0
75%         636.0
max       11488.0
Name: text, dtype: float64

In [14]:
# aired: free-text date ranges. Show the shapes it comes in.
aired_shape = animes["aired"].fillna("<missing>").str.replace(r"\d", "9", regex=True).str.replace(r"[A-Z][a-z]{2}", "Mon", regex=True)
aired_shape.value_counts().head(10)

aired
Mon 99, 9999                    5516
Mon 99, 9999 to Mon 99, 9999    4101
Mon 9, 9999 to Mon 99, 9999     3532
Mon 9, 9999                     2205
9999                            1294
Mon 9, 9999 to Mon 9, 9999       697
Mon 99, 9999 to Mon 9, 9999      587
Mon available                    372
Mon 99, 9999 to ?                265
Mon, 9999                        228
Name: count, dtype: int64

In [15]:
# profiles: categorical values and birthday formats
print(profiles["gender"].value_counts(dropna=False).to_string())
bday_shape = profiles["birthday"].fillna("<missing>").str.replace(r"\d", "9", regex=True).str.replace(r"[A-Z][a-z]{2}", "Mon", regex=True)
print()
print(bday_shape.value_counts().head(8).to_string())

gender
Male          37096
NaN           27871
Female        16018
Non-Binary      742

birthday
<missing>       34920
Mon 99, 9999    25274
Mon 9, 9999     11065
Mon 99           5196
9999             2291
Mon 9            2094
Mon 9999          573
Mon               225


## 5. Numeric ranges

Sanity check for impossible or suspicious values (e.g. zero episodes, scores outside 1–10).

In [16]:
animes[["episodes", "members", "popularity", "ranked", "score"]].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
episodes,18605.0,11.46,47.95,1.00,1.00,2.00,12.00,3057.00
members,19311.0,34726.09,112177.19,25.00,388.00,2389.00,14501.50,1871043.00
popularity,19311.0,7720.83,4676.79,1.00,3725.00,7539.00,11613.00,16338.00
ranked,16099.0,6866.52,4390.02,1.00,2895.50,6963.00,10601.50,14675.00
score,18732.0,6.44,1.01,1.25,5.77,6.41,7.15,9.23


In [17]:
reviews[["score"]].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
score,192112.0,7.57,2.26,0.0,6.0,8.0,9.0,11.0


In [18]:
# MyAnimeList review scores should be 1-10. Count every value, including any outside that range.
reviews.drop_duplicates(subset="uid")["score"].value_counts().sort_index()

score
0         1
1      2916
2      3149
3      5726
4      5880
5      8548
6     12446
7     19064
8     23727
9     24543
10    24518
11        1
Name: count, dtype: int64

## 6. Dictionary skeleton

Writes `docs/data-dictionary.md` with the measured columns filled in and **Meaning** left blank for you to write. It never overwrites an existing file, so your edits are safe if this notebook is re-run.

In [19]:
out = Path("../../docs/data-dictionary.md")

if out.exists():
    print(f"{out} already exists; not overwriting.")
else:
    lines = ["# Data dictionary", "", "Source: Kaggle `marlesson/myanimelist-dataset-animes-profiles-reviews`, version 1.", ""]
    for name, prof in profiles_by_file.items():
        df = raw[name]
        lines += [f"## `{name}.csv` ({len(df):,} rows)", "", "| Column | Type | % Missing | Example | Meaning |", "|---|---|---|---|---|"]
        for col, row in prof.iterrows():
            ex = row["example"].replace("|", "\\|")
            lines.append(f"| `{col}` | {row['dtype']} | {row['missing_%']} | {ex} | |")
        lines.append("")
    out.parent.mkdir(exist_ok=True)
    out.write_text("\n".join(lines))
    print(f"wrote {out.resolve()}")

../../docs/data-dictionary.md already exists; not overwriting.
